## Curado y separación temporal de préstamos

### Datos y variable objetivo

- Se procesaron en Polars los préstamos aceptados de Lending Club.
- Se codificó `loan_status` como `1` para `Default` y `Charged Off`, y `0` para `Fully Paid`. Al excluir los demás estatus, quedaron **1,345,350 préstamos**.

### Curado de variables

- `issue_d` se convirtió a fecha mensual y `addr_state` se transformó mediante *one-hot encoding*.
- `term` se expresó en meses (36 o 60). `emp_length` se codificó de 0 (`< 1 year`) a 10 (`10+ years`) y se conservaron los faltantes.
- Las cinco variables numéricas se conservaron en su escala original.

### Separación temporal

- La corrida actual abarca **junio de 2008 a mayo de 2013**.
- Se eligieron los cierres de mes más cercanos al 70 % y al 90 % acumulado de préstamos. Cada mes se asignó a un solo conjunto y se exportó un CSV por conjunto.
- Los csv se encuentran en la carpeta drive (02_Limpieza-GenSubsets)
- La validación abarca **dos meses**. Queda pendiente ajustar la ventana trimestral prevista para KS y PSI.

### Resultados de la corrida actual

### Separación TEORICA temporal de los préstamos · 5 años

| Conjunto | Desde | Hasta | Meses | % de los 60 meses |
|---|---|---|---:|---:|
| Entrenamiento (`train`) | Jun 2008 | Nov 2011 | 42 | 70 % |
| Prueba (`test`) | Dic 2011 | Nov 2012 | 12 | 20 % |
| Validación (`validation`) | Dic 2012 | May 2013 | 6 | 10 % |
| **Total** | **Jun 2008** | **May 2013** | **60** | **100 %** |

### Separación REAL temporal de los préstamos · 5 años

| Conjunto | Desde | Hasta | Meses | N.º de préstamos | % de préstamos |
|---|---|---|---:|---:|---:|
| Train | Jun 2008 | Dic 2012 | 55 | 92,095 | 68.44 % |
| Test | Ene 2013 | Abr 2013 | 4 | 32,125 | 23.87 % |
| Validation | May 2013 | May 2013 | 1 | 10,350 | 7.69 % |
| **Total** | **Jun 2008** | **May 2013** | **60** | **134,570** | **100.00 %** |

El reparto se define sobre los meses en orden cronológico. Todos los préstamos de un mismo mes pertenecen al mismo conjunto. Para evaluar KS y PSI, la validación se divide en dos ventanas de tres meses consecutivos: diciembre de 2012 a febrero de 2013 y marzo a mayo de 2013.

# Modelo de credito
--

## Acuerdos
- Data de 2008-2012 4 años
- Se evaluaran ventanas de 3 meses
- Col de fecha "issue_d"
- Usar polars
- Variable objetivo "loan_status"
  - Default - 1
  - Fully Paid - 0
  - Charged Off - 1
- Variables explicativas
  - loan_amount
  - emp_lenght
  - dti (deuda ingreso)
  - term
  - int_rate
  - annual_inc
  - addr_state
  - delinq_2yrs
- El modelo a entrenar es una regresion logistica
  - 70 % train (menores a la fecha donde haya 71% de datos)
  - 20 % test (mayores a la fecha donde haya 70% de datos)
  - 10 % validation (donde se calcula el ks y el psi ultimo 10% de datos por fecha)

In [1]:
!pip install polars

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'lending-club' dataset.
Path to dataset files: /kaggle/input/lending-club


In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import polars as pl

In [23]:
import pandas as pd

# Dataset file paths
accepted_path = "/kaggle/input/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv"

#rejected_path = "/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv"

selected_cols = [
    'loan_status',
    'issue_d',
    "loan_amnt",
    "emp_length",
    "dti",
    "term",
    "int_rate",
    "annual_inc",
    "addr_state",
    "delinq_2yrs"
]

credit = pl.read_csv(
    accepted_path,
    schema_overrides={"id": pl.String},
    truncate_ragged_lines=True,
    columns=selected_cols
)




In [24]:
credit['loan_status'].value_counts()


loan_status,count
str,u32
"""Current""",878317
"""Late (31-120 days)""",21467
"""In Grace Period""",8436
"""Late (16-30 days)""",4349
"""Default""",40
"""Charged Off""",268559
null,33
"""Fully Paid""",1076751
"""Does not meet the credit polic…",761


## Limpieza

In [25]:
# crear fechas
credit = credit.with_columns(
    pl.col("issue_d").str.to_date("%b-%Y")
)
credit['issue_d']

issue_d
date
2015-12-01
2015-12-01
2015-12-01
2015-12-01
2015-12-01
…
2016-10-01
2016-10-01
2016-10-01


In [26]:
# Separar variable objetivo
credit = credit.with_columns(
    pl.when(pl.col("loan_status") == "Default")
    .then(1)
    .when(pl.col("loan_status") == "Charged Off")
    .then(1)
    .when(pl.col("loan_status") == "Fully Paid")
    .then(0)
    .alias("loan_status")
)
credit = credit.drop_nulls(subset=['loan_status'])
credit['loan_status'].value_counts(sort=True)

loan_status,count
i32,u32
0,1076751
1,268599


# Separación de Variables categoricas

In [27]:
credit

loan_amnt,term,int_rate,emp_length,annual_inc,issue_d,loan_status,addr_state,dti,delinq_2yrs
f64,str,f64,str,f64,date,i32,str,f64,f64
3600.0,""" 36 months""",13.99,"""10+ years""",55000.0,2015-12-01,0,"""PA""",5.91,0.0
24700.0,""" 36 months""",11.99,"""10+ years""",65000.0,2015-12-01,0,"""SD""",16.06,1.0
20000.0,""" 60 months""",10.78,"""10+ years""",63000.0,2015-12-01,0,"""IL""",10.78,0.0
10400.0,""" 60 months""",22.45,"""3 years""",104433.0,2015-12-01,0,"""PA""",25.37,1.0
11950.0,""" 36 months""",13.44,"""4 years""",34000.0,2015-12-01,0,"""GA""",10.2,0.0
…,…,…,…,…,…,…,…,…,…
18000.0,""" 60 months""",9.49,"""5 years""",130000.0,2016-10-01,0,"""TX""",20.59,0.0
29400.0,""" 60 months""",13.99,"""9 years""",180792.0,2016-10-01,0,"""CA""",22.03,0.0
32000.0,""" 60 months""",14.49,"""3 years""",157000.0,2016-10-01,1,"""AZ""",10.34,0.0


In [28]:
categoricas = ["emp_length", "term", "addr_state"]
numericas = ["loan_amnt", "dti", "int_rate", "annual_inc", "delinq_2yrs"]


In [29]:
credit[categoricas]#.value_counts() #.value_counts(sort=True)

emp_length,term,addr_state
str,str,str
"""10+ years""",""" 36 months""","""PA"""
"""10+ years""",""" 36 months""","""SD"""
"""10+ years""",""" 60 months""","""IL"""
"""3 years""",""" 60 months""","""PA"""
"""4 years""",""" 36 months""","""GA"""
…,…,…
"""5 years""",""" 60 months""","""TX"""
"""9 years""",""" 60 months""","""CA"""
"""3 years""",""" 60 months""","""AZ"""


In [30]:
frecuencias = {}

for variable in categoricas:
    tabla = (
        credit
        .group_by(variable)
        .agg(pl.len().alias("n"))
        .with_columns(
            (100 * pl.col("n") / pl.col("n").sum()).round(2).alias("porcentaje")
        )
        .sort("n", descending=True)
    )
    frecuencias[variable] = tabla

    print(variable)
    with pl.Config(tbl_rows=60):
        display(tabla)

emp_length


emp_length,n,porcentaje
str,u32,f64
"""10+ years""",442209,32.87
"""2 years""",121751,9.05
"""< 1 year""",108065,8.03
"""3 years""",107602,8.0
"""1 year""",88495,6.58
"""5 years""",84154,6.26
"""4 years""",80558,5.99
null,78516,5.84
"""6 years""",62735,4.66


term


term,n,porcentaje
str,u32,f64
""" 36 months""",1020768,75.87
""" 60 months""",324582,24.13


addr_state


addr_state,n,porcentaje
str,u32,f64
"""CA""",196529,14.61
"""TX""",110173,8.19
"""NY""",109849,8.17
"""FL""",95611,7.11
"""IL""",51723,3.84
"""NJ""",48450,3.6
"""PA""",45524,3.38
"""OH""",43844,3.26
"""GA""",43376,3.22


In [31]:
# 1. addr_state: crear un indicador por estado
credit = credit.to_dummies(columns=["addr_state"])

# 2. term: expresar el plazo en meses
credit = credit.with_columns(
    pl.col("term")
    .str.strip_chars()
    .replace_strict({"36 months": 36, "60 months": 60}, return_dtype=pl.UInt8)
)

# 3. emp_length: reportar los faltantes y conservar el orden de los niveles
print(
    credit.select(
        pl.col("emp_length").null_count().alias("n_faltantes"),
        (pl.col("emp_length").is_null().mean() * 100).round(2).alias("pct_faltantes"),
    )
)

orden_emp = {
    "< 1 year": 0,
    "1 year": 1,
    **{f"{i} years": i for i in range(2, 10)},
    "10+ years": 10,
}

credit = credit.with_columns(
    pl.col("emp_length")
    .str.strip_chars()
    .replace_strict(orden_emp, return_dtype=pl.UInt8)
)

shape: (1, 2)
┌─────────────┬───────────────┐
│ n_faltantes ┆ pct_faltantes │
│ ---         ┆ ---           │
│ u32         ┆ f64           │
╞═════════════╪═══════════════╡
│ 78516       ┆ 5.84          │
└─────────────┴───────────────┘


In [32]:
frecuencias = {}
categoricas = ["emp_length", "term"]
for variable in categoricas:
    tabla = (
        credit
        .group_by(variable)
        .agg(pl.len().alias("n"))
        .with_columns(
            (100 * pl.col("n") / pl.col("n").sum()).round(2).alias("porcentaje")
        )
        .sort("n", descending=True)
    )
    frecuencias[variable] = tabla

    print(variable)
    with pl.Config(tbl_rows=60):
        display(tabla)

emp_length


emp_length,n,porcentaje
u8,u32,f64
10,442209,32.87
2,121751,9.05
0,108065,8.03
3,107602,8.0
1,88495,6.58
5,84154,6.26
4,80558,5.99
null,78516,5.84
6,62735,4.66


term


term,n,porcentaje
u8,u32,f64
36,1020768,75.87
60,324582,24.13


# Creacion de conjuntos (70/20/10)

# Cortar set

In [33]:
from datetime import date

credit_completo = credit  # copias

In [34]:
credit.select(
    pl.col("issue_d").min().alias("desde"),
    pl.col("issue_d").max().alias("hasta"),
    pl.len().alias("n_prestamos"),
)

desde,hasta,n_prestamos
date,date,u32
2007-06-01,2018-12-01,1345350


In [35]:
anios = 5
inicio = date(2008, 6, 1)
fin = date(2008 + anios, 6, 1)  # Límite superior excluido

credit = credit_completo.filter(
    (pl.col("issue_d") >= inicio) & (pl.col("issue_d") < fin)
)

tag = f"{inicio.year}_{fin.year}"

In [36]:
credit_completo.select(
    pl.col("issue_d").min().alias("desde"),
    pl.col("issue_d").max().alias("hasta"),
    pl.len().alias("n_prestamos"),
)

desde,hasta,n_prestamos
date,date,u32
2007-06-01,2018-12-01,1345350


In [37]:
credit.select(
    pl.col("issue_d").min().alias("desde"),
    pl.col("issue_d").max().alias("hasta"),
    pl.len().alias("n_prestamos"),
)

desde,hasta,n_prestamos
date,date,u32
2008-06-01,2013-05-01,134570


# Crear subsets

In [38]:
if credit["issue_d"].null_count() > 0:
    raise ValueError("Hay préstamos sin una fecha válida en issue_d. Revisa esas filas.")

In [42]:
'''
#####################
# CORTE TRIMSTRAL #

fecha = pl.col("issue_d")
mes_desde_inicio = (
    (fecha.dt.year() - inicio.year) * 12
    + fecha.dt.month().cast(pl.Int32) - inicio.month
)

por_fecha = (
    credit.group_by((mes_desde_inicio // 3).alias("trimestre"))
    .agg(
        fecha.min().alias("desde"),
        fecha.max().alias("issue_d"),  # cierre para fecha_mas_cercana
        fecha.n_unique().alias("n_meses"),
        pl.len().alias("n_prestamos"),
    )
    .sort("trimestre")
    .with_columns(
        (pl.col("n_prestamos").cum_sum() / pl.col("n_prestamos").sum())
        .alias("fraccion_acumulada")
    )
)

# Comprobar que los 48 meses tienen préstamos en la muestra
assert por_fecha.height == 16 and por_fecha["n_meses"].min() == 3
'''

In [39]:
# por fecha percentil
por_fecha = (
    credit.group_by("issue_d")
    .len(name="n_prestamos")
    .sort("issue_d")
    .with_columns(
        (pl.col("n_prestamos").cum_sum() / pl.col("n_prestamos").sum())
        .alias("fraccion_acumulada")
    )
)


In [40]:
def fecha_mas_cercana(tabla, objetivo):
    return (
        tabla.sort((pl.col("fraccion_acumulada") - objetivo).abs())
        .get_column("issue_d")
        .first()
    )

# Reservar al menos una fecha para test y otra para validation.
corte_70 = fecha_mas_cercana(por_fecha.head(-2), 0.70)

candidatas_90 = por_fecha.head(-1).filter(pl.col("issue_d") > corte_70)
corte_90 = fecha_mas_cercana(candidatas_90, 0.90)

print("Última fecha de train:", corte_70)
print("Última fecha de test: ", corte_90)

Última fecha de train: 2012-12-01
Última fecha de test:  2013-04-01


In [41]:
fecha = pl.col("issue_d")

train = credit.filter(fecha <= corte_70)
test = credit.filter((fecha > corte_70) & (fecha <= corte_90))
validation = credit.filter(fecha > corte_90)

resumen = pl.concat([
    datos.select(
        pl.lit(nombre).alias("conjunto"),
        pl.col("issue_d").min().alias("desde"),
        pl.col("issue_d").max().alias("hasta"),
        pl.len().alias("n_prestamos"),
        (100 * pl.len() / credit.height).round(2).alias("porcentaje"),
    )
    for nombre, datos in [
        ("train", train),
        ("test", test),
        ("validation", validation),
    ]
])

resumen

conjunto,desde,hasta,n_prestamos,porcentaje
str,date,date,u32,f64
"""train""",2008-06-01,2012-12-01,92095,68.44
"""test""",2013-01-01,2013-04-01,32125,23.87
"""validation""",2013-05-01,2013-05-01,10350,7.69


In [42]:
for nombre, datos in [("train", train), ("test", test), ("validation", validation)]:
    archivo = f"credito_{tag}_{nombre}.csv"
    datos.write_csv(archivo)
    print(f"{archivo} | {datos.height:,} préstamos")

credito_2008_2013_train.csv | 92,095 préstamos
credito_2008_2013_test.csv | 32,125 préstamos
credito_2008_2013_validation.csv | 10,350 préstamos


In [24]:
'''
for nombre, datos in [("train", train), ("test", test), ("validation", validation)]:
    desde = datos["issue_d"].min()
    hasta = datos["issue_d"].max()
    archivo = f"credito_{nombre}_{desde:%Y-%m}_{hasta:%Y-%m}.csv"

    datos.write_csv(archivo)
    print(f"{archivo}  |  {datos.height:,} préstamos")
    '''

'\nfor nombre, datos in [("train", train), ("test", test), ("validation", validation)]:\n    desde = datos["issue_d"].min()\n    hasta = datos["issue_d"].max()\n    archivo = f"credito_{nombre}_{desde:%Y-%m}_{hasta:%Y-%m}.csv"\n\n    datos.write_csv(archivo)\n    print(f"{archivo}  |  {datos.height:,} préstamos")\n    '